## Imports

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import mesmer
import model
import model_analysis

import importlib
import matplotlib.pyplot as plt
import xarray as xr

import numpy as np

import cartopy.crs as ccrs
from scipy.stats import beta



: 

## Load Data

In [ ]:
is_local_data = True
Month_idx = 4
safe = True
start = None
end = None
max_depth = 3

In [ ]:
raw_mrsol_for_mean = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=1)

raw_mrsol_for_var = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=2)

raw_mrsol_for_skew = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=3)

In [ ]:
raw_mrsol_empirical_maximas_mean = raw_mrsol_for_mean.max("time")
raw_mrsol_empirical_maximas_var = raw_mrsol_for_var.max("time")
raw_mrsol_empirical_maximas_skew = raw_mrsol_for_skew.max("time")

raw_mrsol_empirical_maximas = xr.concat([raw_mrsol_empirical_maximas_mean,raw_mrsol_empirical_maximas_var,raw_mrsol_empirical_maximas_skew],dim="sets").max("sets")


In [ ]:

max_v = raw_mrsol_empirical_maximas.copy(deep=True)
max_v["mrsol"] = (max_v["mrsol"] * 1.1).clip(min=1e-5)
approx_maximas = max_v


In [ ]:
tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=1)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=1)
tas_ds = model.shape_data.prune_group_ds_timespan(tas_ds,Month_idx=Month_idx)
pr_ds = model.shape_data.prune_group_ds_timespan(pr_ds,Month_idx=Month_idx)

raw_input_for_mean = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=2)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=2)
tas_ds = model.shape_data.prune_group_ds_timespan(tas_ds,Month_idx=Month_idx)
pr_ds = model.shape_data.prune_group_ds_timespan(pr_ds,Month_idx=Month_idx)

raw_input_for_var = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=3)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=3)
tas_ds = model.shape_data.prune_group_ds_timespan(tas_ds,Month_idx=Month_idx)
pr_ds = model.shape_data.prune_group_ds_timespan(pr_ds,Month_idx=Month_idx)

raw_input_for_skew = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=4)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=4)
tas_ds = model.shape_data.prune_group_ds_timespan(tas_ds,Month_idx=Month_idx)
pr_ds = model.shape_data.prune_group_ds_timespan(pr_ds,Month_idx=Month_idx)

raw_input_for_test = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")


some explantions
slice 0-4 weil im moment die letzte schicht hartnäckig probleme macht

In [ ]:
def shape_target(ds, maximas):
    ds = ds.clip(min = 0)
    ds = (ds/maximas).clip(max = 1-1e-15)
    ds = ds.sel(time=slice(start, end)).resample(time="ME").mean()
    ds = ds.sel(time= ds.time.dt.month == Month_idx).isel(depth=slice(0, max_depth))
    return ds

In [ ]:
def mask_stack_target(ds):
    masked_ds, chunk_mask, detail_mask= model.mask.mask_nonpositiv_height_chunks(ds.drop_vars("depth_bnds"))
    ds = mesmer.grid.stack_lat_lon(masked_ds)
    ds = ds.transpose("gridcell","time", "depth")
    return ds,chunk_mask, detail_mask

In [ ]:
# Transform already as one function
#model.transform.Logit_Transform_ds()

In [ ]:
def shape_input(ds, chunk_mask):
    ds = ds.sel(time=slice(start, end)).resample(time="ME").mean()
    ds = ds.sel(time= ds.time.dt.month == Month_idx)
    ds = model.mask.mask_mask(ds,chunk_mask)
    ds = mesmer.grid.stack_lat_lon(ds)
    return ds

    

In [ ]:
#The chunk_mask schould all be the same, this is important that there are no shape problems in the regressions, you can test this with  
#(chunk_mask != chunk_mask_2).sum(),#(chunk_mask != chunk_mask_1).sum(),
#The chunk_mask is used, that the regression can predict a depth.size vector at each gridcell, the masks: detail_mask_mean, detail_mask_var and detail_mask_test are later used to determan which points are real predictions and witch where just placeholders to let the regression run smoothly.

### Linear Regression of the mean

In [ ]:
mean_target_noneT = shape_target(raw_mrsol_for_mean, approx_maximas)
mean_target_noneT, chunk_mask, detail_mask = mask_stack_target(mean_target_noneT)
mean_target = model.transform.Logit_Transform_ds(mean_target_noneT)
mean_target_da = mean_target.mrsol

In [ ]:
mean_predictors = shape_input(raw_input_for_mean,chunk_mask)

In [ ]:
Regr_set_mean = {}

In [ ]:
#mean_target_da

In [ ]:
mean_predictors

In [ ]:
for i,key in enumerate(["mean_1","mean_2"], start=1):
    Regr_set_mean[key] = model.stats._parallel_polynomial_regression.ParPolyRegression(degree=i)
    Regr_set_mean[key].fit(predictors=mean_predictors, target=mean_target_da,location_dim="gridcell", regr_dim="time")

### Compute Residuals for the variance

Hier wäre die meinung einen neuen Run zu verwenden, die frage ist ob sich die variance durch das fehlen des runs zu tief ausfällt. Overfitting korrektur mit 1/(1-param/n_samples)^2? (SPäter probieren)

In [ ]:
var_target_noneT = shape_target(raw_mrsol_for_var, approx_maximas)
var_target_noneT, chunk_mask_var, detail_mask_var= mask_stack_target(var_target_noneT)#Hier können noch sehr grosse werte auftauchen, wenn in irgendwelchen schichten die Maximas der verschieden runs sehr unterschiedlich sind.

var_target = model.transform.Logit_Transform_ds(var_target_noneT)



In [ ]:
var_predictors = shape_input(raw_input_for_var,chunk_mask)

In [ ]:
residuals = {}
for key, regr in Regr_set_mean.items():
    residuals[key] = regr.residuals(var_predictors, var_target)


### Linear Regression of the Variance

In [ ]:
Regr_set_var = {}

In [ ]:
for mean_key, regr in Regr_set_mean.items():
    for i,var_key in enumerate(["var_1","var_2"]):
        Regr_set_var[f"{mean_key}_{var_key}"] = model.stats._parallel_polynomial_regression.ParPolyRegression(degree=i+1)
        Regr_set_var[f"{mean_key}_{var_key}"].fit(predictors=var_predictors, target=(residuals[mean_key].residuals)**2,location_dim="gridcell", regr_dim="time")

for simplicity not in use
### Compute skewness samples
skew_target_noneT = shape_target(raw_mrsol_for_skew, approx_maximas)
skew_target_noneT, chunk_mask_skew, detail_mask_skew = mask_stack_target(skew_target_noneT)
skew_target = model.transform.Logit_Transform_ds(skew_target_noneT)

skew_predictors = shape_input(raw_input_for_skew,chunk_mask)
mean_prediction = LinReg_mean.predict(skew_predictors)
residuals = skew_target.mrsol - mean_prediction.prediction
sigmas = np.sqrt(LinReg_variance.predict(skew_predictors).prediction.clip(min = 1e-32))
standardized_values_for_skew = (residuals/sigmas)
### Linear Regression of the Skewness
LinReg_skewness = model.stats._parallel_linear_regression.ParLinearRegression()
LinReg_skewness.fit(predictors=skew_predictors, target=(standardized_values_for_skew)**3,location_dim="gridcell", regr_dim="time")

### Predictor

### Export Prameters

In [ ]:
if safe:
    for mean_key, mean_regr in Regr_set_mean.items():
        for var_key in ["var_1","var_2"]:
            model.save.save_params(mean_regr.params,Regr_set_var[f"{mean_key}_{var_key}"].params, maximas=approx_maximas,chunk_mask=chunk_mask,detail_mask=detail_mask,var = "mrsol",scen= "historical", folder = f"MPI-ESM1-2-LR/variations/mean_var_regr_degree/{mean_key}_{var_key}/local{is_local_data}/month{Month_idx}", name=f"start={start},end={end}.max_depth{max_depth}")